In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

from sklearn.cluster import KMeans, AffinityPropagation

# --- PASSO 1: DIVISÃO DOS DADOS ---
X_clust, _ = make_blobs(n_samples=1000, centers=4, n_features=5, random_state=42)

X_train_full, X_test = train_test_split(X_clust, test_size=0.20, random_state=42)
X_train, X_val = train_test_split(X_train_full, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

X_train_full_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

models_clust = {
    "K-Means": {
        "class": KMeans,
        "params": [{"n_clusters": k, "random_state": 42, "n_init": 10} for k in [2, 3, 4, 5]]
    },
    "Affinity Propagation": {
        "class": AffinityPropagation,
        "params": [{"damping": d, "random_state": 42} for d in [0.5, 0.7, 0.9]]
    }
}

res_train_cl, res_val_cl, res_test_cl = [], [], []

for name, config in models_clust.items():
    # --- PASSOS 2, 3 e 4: MODELO DEFAULT ---
    default_model = config["class"]()
    tr_labels_def = default_model.fit_predict(X_train_scaled)
    val_labels_def = default_model.predict(X_val_scaled) if hasattr(default_model, "predict") else default_model.fit_predict(X_val_scaled)
    
    # Passo 3: Treino (Default)
    res_train_cl.append({"Algoritmo": name, "Silhouette Score": round(silhouette_score(X_train_scaled, tr_labels_def), 4)})
    
    # Passo 4: Validação (Default)
    res_val_cl.append({"Algoritmo": name, "Silhouette Score": round(silhouette_score(X_val_scaled, val_labels_def), 4)})
    
    # --- PASSO 5: BUSCA PELO MELHOR HIPERPARÂMETRO ---
    best_score = -1
    best_p = None
    
    for p in config["params"]:
        model = config["class"](**p)
        train_labels = model.fit_predict(X_train_scaled)
        
        if len(set(train_labels)) < 2:
            continue
            
        val_labels = model.predict(X_val_scaled) if hasattr(model, "predict") else model.fit_predict(X_val_scaled)
        
        if len(set(val_labels)) < 2:
            continue
            
        val_score = silhouette_score(X_val_scaled, val_labels)
        
        if val_score > best_score:
            best_score = val_score
            best_p = p
            
    # --- PASSOS 6 e 7: RETREINAMENTO (TREINO + VALIDAÇÃO) ---
    final_model = config["class"](**best_p)
    if hasattr(final_model, "predict"):
        final_model.fit(X_train_full_scaled)
        test_labels = final_model.predict(X_test_scaled)
    else:
        test_labels = final_model.fit_predict(X_test_scaled)
        
    # --- PASSO 8: PERFORMANCE NO TESTE ---
    test_score = silhouette_score(X_test_scaled, test_labels)
    res_test_cl.append({"Algoritmo": name, "Silhouette Score": round(test_score, 4)})

# Exibição das Tabelas do Ensaio
print("=== 1) AGRUPAMENTO: DADOS DE TREINO (DEFAULT) ===")
print(pd.DataFrame(res_train_cl).to_string(index=False))

print("\n=== 2) AGRUPAMENTO: DADOS DE VALIDAÇÃO (DEFAULT) ===")
print(pd.DataFrame(res_val_cl).to_string(index=False))

print("\n=== 3) AGRUPAMENTO: DADOS DE TESTE (MODELO OTIMIZADO) ===")
print(pd.DataFrame(res_test_cl).to_string(index=False))

=== 1) AGRUPAMENTO: DADOS DE TREINO (DEFAULT) ===
           Algoritmo  Silhouette Score
             K-Means            0.4499
Affinity Propagation            0.1921

=== 2) AGRUPAMENTO: DADOS DE VALIDAÇÃO (DEFAULT) ===
           Algoritmo  Silhouette Score
             K-Means            0.4746
Affinity Propagation            0.1769

=== 3) AGRUPAMENTO: DADOS DE TESTE (MODELO OTIMIZADO) ===
           Algoritmo  Silhouette Score
             K-Means            0.7283
Affinity Propagation            0.1892
